In [1]:
# %%bash
# pip install -U huggingface_hub

# python -m huggingface_hub.cli download xlangai/AgentNet \
#   --repo-type dataset \
#   --include "win_mac_images/images.z0[1-3]" \
#   --local-dir /content/drive/MyDrive/AgentNet



In [1]:
import os
from google.colab import drive
import json
import re
from collections import Counter
from tqdm import tqdm
from collections import defaultdict



drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
repo_id = "xlangai/AgentNet"
base_dir = "/content/drive/MyDrive/AgentNet"


img_dir = f"{base_dir}/win_mac_images"
zip_path = f"{img_dir}/images.zip"
extract_dir = os.path.join(img_dir, "office_images")
task_id_filtered = f"{base_dir}/office_task_ids.json"
task_clean = f"{base_dir}/office_task_ids_clean.json"
traj_file = f"{base_dir}/agentnet_win_mac_18k.jsonl"


os.makedirs(img_dir, exist_ok=True)
print("Target directory:", img_dir)


Target directory: /content/drive/MyDrive/AgentNet/win_mac_images


In [4]:
# with open(task_id_filtered, "r", encoding="utf-8") as f:
#     task_ids = json.load(f)

# pattern = re.compile(r"^\d{14}_[0-9a-fA-F\-]{36}$")

# clean_ids = []
# bad_ids = []

# for tid in task_ids:
#     if pattern.match(tid):
#         clean_ids.append(tid)
#     else:
#         bad_ids.append(tid)

# print("Valid task_ids:", len(clean_ids))
# print("Invalid entries:", len(bad_ids))

# print("\nBad entries:")
# for x in bad_ids[:10]:
#     print(x)

# clean_path = "/content/drive/MyDrive/AgentNet/office_task_ids_clean.json"

# with open(clean_path, "w", encoding="utf-8") as f:
#     json.dump(clean_ids, f)

# print("Clean task list saved:", clean_path)

In [3]:
with open(task_clean, "r", encoding="utf-8") as f:
    office_task_ids = set(json.load(f))

print("Office tasks:", len(office_task_ids))

Office tasks: 2450


### Extract office images (if needed)

In [ ]:
office_images = set()

with open(traj_file, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)

        if obj.get("task_id") in office_task_ids:
            for step in obj.get("traj", []):
                img = step.get("image")
                if img:
                    office_images.add(img)

print("Office screenshots:", len(office_images))
print("Sample:", list(office_images)[:5])

Office screenshots: 46118
Sample: ['bec864fe-e2e4-4dc7-aec5-a4ab5c7fb385.png', 'b7fff8fd-3cf2-4ddf-bb35-566302e56968.png', '98ec4599-a406-4214-b960-044f8370c58c.png', '47064f2b-8269-4747-9b21-4b84b2c20013.png', '75ab0c4a-e2c4-4419-837b-4111b21b7406.png']


### Actions Statistic

In [7]:
action_counter = Counter()

with open(traj_file, "r", encoding="utf-8") as f:

    for line in f:

        data = json.loads(line)

        for step in data["traj"]:

            code = step["value"]["code"]

            matches = re.findall(r'pyautogui\.([a-zA-Z_]+)', code)

            for action in matches:
                action_counter[action] += 1

            if "terminate" in code:
                action_counter["terminate"] += 1


print("Unique actions:", len(action_counter))
print(sorted(action_counter.keys()))
print(action_counter)

Unique actions: 12
['click', 'doubleClick', 'dragTo', 'hotkey', 'hscroll', 'middleClick', 'moveTo', 'press', 'rightClick', 'scroll', 'terminate', 'write']
Counter({'click': 235648, 'moveTo': 35709, 'write': 31118, 'press': 22308, 'scroll': 19142, 'terminate': 17564, 'dragTo': 16273, 'doubleClick': 8168, 'hotkey': 7182, 'rightClick': 4273, 'hscroll': 546, 'middleClick': 42})


In [30]:
def get_action_statistics(file_path):
    # Initialize a counter to keep track of action frequencies
    action_counter = Counter()
    total_actions = 0
    error_count = 0
    
    print(f"Loading dataset from {file_path}...")
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            dataset = json.load(f)
    except FileNotFoundError:
        print(f"Error: Could not find the file '{file_path}'.")
        return

    print("Analyzing action types...\n")
    
    # Loop through every sample in the dataset
    for item in dataset:
        for message in item.get('conversations', []):
            # We only care about the assistant's response
            if message.get('from') == 'assistant':
                value_str = message.get('value', '')
                try:
                    # Parse the nested JSON string inside 'value'
                    action_data = json.loads(value_str)
                    action_type = action_data.get('ACTION')
                    
                    if action_type:
                        action_counter[action_type] += 1
                        total_actions += 1
                except json.JSONDecodeError:
                    error_count += 1
                    print(f"Warning: Could not parse action JSON in item ID: {item.get('id')}")

    # Print the final statistics
    print("-" * 30)
    print("🎯 ACTION TYPE STATISTICS")
    print("-" * 30)
    
    if total_actions == 0:
        print("No actions found.")
        return

    # Loop through the counted actions, sorted by most common
    for action, count in action_counter.most_common():
        percentage = (count / total_actions) * 100
        print(f"{action:<15} | {count:>6} ({percentage:>5.1f}%)")
        
    print("-" * 30)
    print(f"Total valid actions parsed: {total_actions}")
    if error_count > 0:
        print(f"Failed to parse: {error_count} items")

# Replace 'your_dataset.json' with your actual file name
file_path = f"{base_dir}/excel/excel_windows_mind2web_style.json"
get_action_statistics(file_path)
file_path = f"{base_dir}/powerpoint/powerpoint_windows_mind2web_style.json"
get_action_statistics(file_path)
file_path = f"{base_dir}/word/word_windows_mind2web_style_testType.json"
get_action_statistics(file_path)

Loading dataset from /content/drive/MyDrive/AgentNet/excel/excel_windows_mind2web_style.json...
Analyzing action types...

------------------------------
🎯 ACTION TYPE STATISTICS
------------------------------
CLICK           |   4251 ( 63.5%)
TYPE            |    703 ( 10.5%)
PRESS           |    508 (  7.6%)
DRAG            |    482 (  7.2%)
TERMINATE       |    260 (  3.9%)
DOUBLE_CLICK    |    177 (  2.6%)
RIGHT_CLICK     |    136 (  2.0%)
HOTKEY          |    128 (  1.9%)
SCROLL          |     46 (  0.7%)
------------------------------
Total valid actions parsed: 6691
Loading dataset from /content/drive/MyDrive/AgentNet/powerpoint/powerpoint_windows_mind2web_style.json...
Analyzing action types...

------------------------------
🎯 ACTION TYPE STATISTICS
------------------------------
CLICK           |   3755 ( 71.1%)
DRAG            |    407 (  7.7%)
TYPE            |    317 (  6.0%)
TERMINATE       |    206 (  3.9%)
PRESS           |    200 (  3.8%)
RIGHT_CLICK     |    166 (  3.

### Office Images folder Statistic

In [ ]:
# 1. Define your paths here
json_file_path = f"{base_dir}/excel_windows_mind2web_style.json"  # Replace with the name of your JSON file
image_dir = f"{base_dir}/office_images"  # Replace with the name of your image directory

def verify_images():
    print("Loading dataset...")
    # Load the JSON file
    try:
        with open(json_file_path, 'r', encoding='utf-8') as f:
            dataset = json.load(f)
    except FileNotFoundError:
        print(f"Error: Could not find the file '{json_file_path}'.")
        return

    print(f"Found {len(dataset)} samples. Checking image folder...")

    matched_count = 0
    missing_count = 0
    missing_images = []

    # 2. Check each image
    for sample in dataset:
        image_name = sample.get('image')
        
        if not image_name:
            continue # Skip if the sample doesn't have an image field
            
        # Construct the full path to where the image SHOULD be
        image_path = os.path.join(image_dir, image_name)
        
        # 3. Verify existence
        if os.path.isfile(image_path):
            matched_count += 1
        else:
            missing_count += 1
            missing_images.append(image_name)

    # 4. Print the final report
    print("-" * 30)
    print("VERIFICATION COMPLETE")
    print("-" * 30)
    print(f"Total samples checked: {matched_count + missing_count}")
    print(f"✅ Images Found:     {matched_count}")
    print(f"❌ Images Missing:   {missing_count}")
    
    # Optionally save the list of missing images to a file so you can investigate
    if missing_count > 0:
        missing_log_path = 'missing_images_log.txt'
        with open(missing_log_path, 'w', encoding='utf-8') as f:
            for img in missing_images:
                f.write(f"{img}\n")
        print(f"\nA list of the missing images has been saved to: {missing_log_path}")


verify_images()

Loading dataset...
Found 3671 samples. Checking image folder...
------------------------------
VERIFICATION COMPLETE
------------------------------
Total samples checked: 3671
✅ Images Found:     3671
❌ Images Missing:   0


## Filter samples from List of "task_id"

In [18]:
# 1. Define your file paths here
ids_file_path = f"{base_dir}/taskID_powerpoint.json"  # Your massive dataset file
output_file_path = f"{base_dir}/powerpoint _windows_samples.jsonl" # Where to save the results

def filter_dataset():
    print("Loading valid task IDs...")
    with open(ids_file_path, 'r', encoding='utf-8') as f:
        id_data = json.load(f)
        valid_task_ids = {item['task_id'] for item in id_data}
    
    print(f"Found {len(valid_task_ids)} unique target IDs. Filtering dataset...")
    
    match_count = 0
    seen_ids = set() # Keeping the deduplication memory
    
    with open(traj_file, 'r', encoding='utf-8') as infile, \
         open(output_file_path, 'w', encoding='utf-8') as outfile:
        
        for line in infile:
            if not line.strip():
                continue 
                
            sample = json.loads(line)
            task_id = sample.get('task_id')
            
            # Check if it's a valid ID AND we haven't seen it yet
            if task_id in valid_task_ids and task_id not in seen_ids:
                
                # Write the JSON object as a single line, followed by a newline
                json.dump(sample, outfile, ensure_ascii=False)
                outfile.write('\n')
                
                seen_ids.add(task_id) # Mark this ID as "saved"
                match_count += 1
                
    print(f"Done! Successfully saved {match_count} unique samples in JSONL format to {output_file_path}")


filter_dataset()

Loading valid task IDs...
Found 267 unique target IDs. Filtering dataset...
Done! Successfully saved 267 unique samples in JSONL format to /content/drive/MyDrive/AgentNet/powerpoint _windows_samples.jsonl


## Mapping to Mind2Web-SoM format

### helping funcs

In [14]:
# action_map = {
#     "click": "CLICK",
#     "doubleClick": "DOUBLE_CLICK",
#     "rightClick": "RIGHT_CLICK",
#     "middleClick": "MIDDLE_CLICK",
#     "moveTo": "MOVE",
#     "dragTo": "DRAG",
#     "scroll": "SCROLL",
#     "hscroll": "HSCROLL",
#     "write": "TYPE",
#     "press": "PRESS",
#     "hotkey": "HOTKEY",
# }

# Cache to reduce API calls
history_cache = {}

def extract_history(action_text):
    """
    Convert AgentNet natural action text to a short history line
    without calling any external model.
    """

    text = action_text.strip()

    # remove quotes around UI labels
    text = re.sub(r'"([^"]+)"', r'\1', text)

    # normalize verbs to uppercase
    replacements = {
        "click": "CLICK",
        "double click": "DOUBLE_CLICK",
        "right click": "RIGHT_CLICK",
        "middle click": "MIDDLE_CLICK",
        "type": "TYPE",
        "press": "PRESS",
        "scroll": "SCROLL",
        "drag": "DRAG",
        "move": "MOVE",
        "write": "TYPE"
    }

    lower = text.lower()

    for k, v in replacements.items():
        if lower.startswith(k):
            return text.replace(text.split()[0], v)

    return text


def parse_code(code, verbose=True):
    actions = []

    # -------------------------
    # Robust unified pattern (order preserved)
    # - hscroll BEFORE scroll (important)
    # - write supports nested parentheses
    # -------------------------
    pattern = re.compile(
        r"(moveTo\(.*?\)|dragTo\(.*?\)|doubleClick\(.*?\)|rightClick\(.*?\)|middleClick\(.*?\)|click\(.*?\)|write\((?:[^)(]+|\([^)(]*\))*\)|press\(.*?\)|hotkey\(.*?\)|hscroll\(.*?\)|scroll\(.*?\))",
        re.DOTALL | re.IGNORECASE
    )

    for match in pattern.finditer(code):
        stmt = match.group(0).strip()
        stmt_lower = stmt.lower()

        parsed = False  # 🔴 track success

        # -------------------------
        # MOVE
        # -------------------------
        if stmt_lower.startswith("moveto"):
            xy = re.search(r"x\s*=\s*([\d\.]+).*?y\s*=\s*([\d\.]+)", stmt, re.IGNORECASE)
            if xy:
                actions.append({
                    "ACTION": "MOVE",
                    "MARK": [float(xy.group(1)), float(xy.group(2))],
                    "VALUE": None
                })
                parsed = True

        # -------------------------
        # DRAG
        # -------------------------
        elif stmt_lower.startswith("dragto"):
            xy = re.search(r"x\s*=\s*([\d\.]+).*?y\s*=\s*([\d\.]+)", stmt, re.IGNORECASE)
            if xy:
                actions.append({
                    "ACTION": "DRAG",
                    "MARK": [float(xy.group(1)), float(xy.group(2))],
                    "VALUE": None
                })
                parsed = True

        # -------------------------
        # CLICK variants
        # -------------------------
        elif any(stmt_lower.startswith(k) for k in ["click", "doubleclick", "rightclick", "middleclick"]):
            action_map = {
                "click": "CLICK",
                "doubleclick": "DOUBLE_CLICK",
                "rightclick": "RIGHT_CLICK",
                "middleclick": "MIDDLE_CLICK"
            }

            action_name = None
            for key in action_map:
                if stmt_lower.startswith(key):
                    action_name = action_map[key]
                    break

            xy = re.search(r"x\s*=\s*([\d\.]+).*?y\s*=\s*([\d\.]+)", stmt, re.IGNORECASE)
            mark = [float(xy.group(1)), float(xy.group(2))] if xy else None

            actions.append({
                "ACTION": action_name,
                "MARK": mark,
                "VALUE": None
            })
            parsed = True

        # -------------------------
        # TYPE (robust, tolerant)
        # -------------------------
        elif stmt_lower.startswith("write"):
            value = None

            # Case 1: proper quoted string (message= / text= / content= / positional)
            m = re.search(
                r"write\(\s*(?:message|text|content)?\s*=?\s*['\"]([\s\S]*?)['\"]",
                stmt,
                re.IGNORECASE
            )

            if m:
                value = m.group(1).strip()

            # Case 2: broken string / missing quote
            if value is None:
                m = re.search(
                    r"write\(\s*(?:message|text|content)?\s*=?\s*([^\)]*)",
                    stmt,
                    re.IGNORECASE
                )
                if m:
                    value = m.group(1).strip()

                    # cleanup noise
                    value = value.strip()
                    value = value.rstrip(')')
                    value = value.strip("'\"")

            if value:
                actions.append({
                    "ACTION": "TYPE",
                    "MARK": None,
                    "VALUE": value
                })
                parsed = True

        # -------------------------
        # PRESS
        # -------------------------
        elif stmt_lower.startswith("press"):
            m = re.search(r"press\(['\"](.*?)['\"]", stmt, re.IGNORECASE)
            if m:
                actions.append({
                    "ACTION": "PRESS",
                    "MARK": None,
                    "VALUE": m.group(1)
                })
                parsed = True

        # -------------------------
        # HOTKEY
        # -------------------------
        elif stmt_lower.startswith("hotkey"):
            keys = re.findall(r"['\"](.*?)['\"]", stmt)
            if keys:
                actions.append({
                    "ACTION": "HOTKEY",
                    "MARK": None,
                    "VALUE": keys
                })
                parsed = True

        # -------------------------
        # HSCROLL (must be before scroll)
        # -------------------------
        elif stmt_lower.startswith("hscroll"):
            m = re.search(r"hscroll\(([-\d]+)\)", stmt, re.IGNORECASE)
            if m:
                actions.append({
                    "ACTION": "HSCROLL",
                    "MARK": None,
                    "VALUE": int(m.group(1))
                })
                parsed = True

        # -------------------------
        # SCROLL
        # -------------------------
        elif stmt_lower.startswith("scroll"):
            m = re.search(r"scroll\(([-\d]+)\)", stmt, re.IGNORECASE)
            if m:
                actions.append({
                    "ACTION": "SCROLL",
                    "MARK": None,
                    "VALUE": int(m.group(1))
                })
                parsed = True

        # -------------------------
        # GLOBAL FALLBACK WARNING
        # -------------------------
        if not parsed and verbose:
            print(f"⚠️ Failed to parse action: {stmt}")

    return actions

def merge_actions(actions):
    merged = []
    current_pos = None

    i = 0
    while i < len(actions):
        act = actions[i]

        if act["ACTION"] == "MOVE":
            current_pos = act["MARK"]
            i += 1
            continue

        # -------------------------
        # DRAG (needs start + end)
        # -------------------------
        if act["ACTION"] == "DRAG":
            if current_pos:
                merged.append({
                    "ACTION": "DRAG",
                    "MARK": [current_pos, act["MARK"]],
                    "VALUE": None
                })
            else:
                # fallback (no start known)
                merged.append({
                    "ACTION": "DRAG",
                    "MARK": [None, act["MARK"]],
                    "VALUE": None
                })
            current_pos = act["MARK"]
            i += 1
            continue

        # -------------------------
        # SCROLL (needs anchor)
        # -------------------------
        if act["ACTION"] == "SCROLL":
            merged.append({
                "ACTION": "SCROLL",
                "MARK": current_pos,  # may be None
                "VALUE": act["VALUE"]
            })
            i += 1
            continue

        # -------------------------
        # CLICK variants
        # -------------------------
        if act["ACTION"] in ["CLICK", "DOUBLE_CLICK", "RIGHT_CLICK", "MIDDLE_CLICK"]:
            merged.append(act)
            current_pos = act["MARK"]
            i += 1
            continue

        # -------------------------
        # TYPE / PRESS / HOTKEY
        # -------------------------
        merged.append(act)
        i += 1

    return merged


def build_prompt(instruction, history):

    history_text = "\n".join(f"- {h}" for h in history) if history else "None"

    prompt = f"""<image>
Imagine that you are imitating humans doing GUI navigation step by step.

You can perform actions such as CLICK, DOUBLE_CLICK, RIGHT_CLICK, MIDDLE_CLICK, MOVE, DRAG, SCROLL, HSCROLL, TYPE, PRESS, HOTKEY.

Output format must be:
{{"ACTION": action_type, "MARK": numeric_id, "VALUE": text_or_null}}

Task: {instruction}

Previous actions:
{history_text}

For your convenience, UI elements are labeled with numeric marks.

What is the next action?
"""

    return prompt

### Running script

In [33]:
import json
from collections import defaultdict
from tqdm import tqdm

with open(task_clean) as f:
    office_task_ids = set(json.load(f))

print("Filtered office tasks:", len(office_task_ids))

dataset = []
matched_tasks = 0
sample_id = 0
action_stats = defaultdict(int)



input_file = f"{base_dir}/word/word_windows_samples.jsonl"
output_file = f"{base_dir}/word/word_windows_mind2web_style.json"


with open(input_file, "r") as f:

    for line in tqdm(f):
        data = json.loads(line)
        task_id = data["task_id"]

        if task_id not in office_task_ids:
            continue

        matched_tasks += 1
        instruction = data["instruction"]
        history = []

        traj = data.get("traj", [])

        for step in traj:

            value = step.get("value", {})

            # 🔥 safety check
            if not isinstance(value, dict):
                continue

            if not value.get("last_step_correct", True):
                continue

            action_text = value.get("action", "")
            code = value.get("code", "")
            image_name = step.get("image")

            parsed_actions = parse_code(code)

            merged_actions = merge_actions(parsed_actions)

            for act in merged_actions:

                action_stats[act["ACTION"]] += 1

                user_prompt = build_prompt(instruction, history)

                sample = {
                    "id": f"AgentNet_SoM_{sample_id}",
                    "image": image_name,
                    "conversations": [
                        {"from": "user", "value": user_prompt},
                        {"from": "assistant", "value": json.dumps(act)}
                    ]
                }

                dataset.append(sample)
                sample_id += 1

                # ✅ update history AFTER sample
                history_line = extract_history(action_text)
                history.append(history_line)

        # ------------------------------------------------
        # ✅ ADD TERMINATE 
        # ------------------------------------------------
        if data.get("task_completed", False) and traj:

            image_name = traj[-1].get("image")

            user_prompt = build_prompt(instruction, history)

            terminate_action = {
                "ACTION": "TERMINATE",
                "MARK": None,
                "VALUE": None
            }

            dataset.append({
                "id": f"AgentNet_SoM_{sample_id}",
                "image": image_name,
                "conversations": [
                    {"from": "user", "value": user_prompt},
                    {"from": "assistant", "value": json.dumps(terminate_action)}
                ]
            })

            sample_id += 1


# ------------------------------------------------
# Save dataset
# ------------------------------------------------
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)

print("\nConversion complete.")
print("Matched tasks:", matched_tasks)
print("Total training samples:", len(dataset))
print(action_stats)

Filtered office tasks: 2450


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/AgentNet/word/word_windows_samples.jsonl'